In [2]:
import pandas as pd
import requests
import time

In [ ]:
url = "https://dapi.kakao.com/v2/local/search/keyword.json"
headers = {"Authorization": "KakaoAK YOUR_KAKAO_API_KEY"} 
params = {
    "query": "서울 아파트"
}

res = requests.get(url, headers=headers, params=params, timeout=5)

res.json()['documents'][0]

{'address_name': '서울 강남구 대치동 316',
 'category_group_code': '',
 'category_group_name': '',
 'category_name': '부동산 > 주거시설 > 아파트',
 'distance': '',
 'id': '11335658',
 'phone': '',
 'place_name': '은마아파트',
 'place_url': 'http://place.map.kakao.com/11335658',
 'road_address_name': '서울 강남구 삼성로 212',
 'x': '127.06532735974666',
 'y': '37.49741836284779'}

In [14]:
url = "https://dapi.kakao.com/v2/local/search/category.json"
headers = {"Authorization": "KakaoAK YOUR_KAKAO_API_KEY"} 
params = {
    "category_group_code": "MT1"
}

res = requests.get(url, headers=headers, params=params, timeout=5)

# res.json()['documents'][0]
for i in range(len(res.json()['documents'])):
    print(res.json()['documents'][i])

{'address_name': '서울 은평구 응암동 90-1', 'category_group_code': 'MT1', 'category_group_name': '대형마트', 'category_name': '가정,생활 > 대형마트 > 이마트', 'distance': '', 'id': '21128108', 'phone': '02-380-1234', 'place_name': '이마트 은평점', 'place_url': 'http://place.map.kakao.com/21128108', 'road_address_name': '서울 은평구 은평로 111', 'x': '126.920178530965', 'y': '37.6002746471855'}
{'address_name': '서울 양천구 목동 962', 'category_group_code': 'MT1', 'category_group_name': '대형마트', 'category_name': '가정,생활 > 대형마트 > 이마트', 'distance': '', 'id': '8047162', 'phone': '02-6923-1234', 'place_name': '이마트 목동점', 'place_url': 'http://place.map.kakao.com/8047162', 'road_address_name': '서울 양천구 오목로 299', 'x': '126.8702985523677', 'y': '37.52556123754674'}
{'address_name': '서울 중구 황학동 2545', 'category_group_code': 'MT1', 'category_group_name': '대형마트', 'category_name': '가정,생활 > 대형마트 > 이마트', 'distance': '', 'id': '8533171', 'phone': '02-2290-1234', 'place_name': '이마트 청계천점', 'place_url': 'http://place.map.kakao.com/8533171', 'road_addre

In [43]:
import requests
import pandas as pd
import time

url = "https://dapi.kakao.com/v2/local/search/category.json"
headers = {"Authorization": "KakaoAK YOUR_KAKAO_API_KEY"}

# 예시 bbox
# 형식: (min_x, min_y, max_x, max_y)
regions = {
    "서울":   (126.764, 37.425, 127.183, 37.701),
    "인천":   (126.360, 37.370, 126.800, 37.780),
    "성남":   (127.060, 37.350, 127.190, 37.470),
    "고양":   (126.730, 37.620, 126.970, 37.750),
    "부천":   (126.730, 37.470, 126.850, 37.560),
    "광명":   (126.830, 37.400, 126.910, 37.470),
    "하남":   (127.120, 37.500, 127.270, 37.620),
    "남양주": (127.120, 37.550, 127.400, 37.780),
    "용인":   (127.000, 37.080, 127.350, 37.350),
    "김포":   (126.600, 37.560, 126.820, 37.750),
    "구리":   (127.080, 37.560, 127.170, 37.640),
    "안양":   (126.900, 37.350, 127.020, 37.440),
}

def split_bbox(min_x, min_y, max_x, max_y, nx=2, ny=2):
    """bbox를 nx * ny 격자로 분할"""
    x_step = (max_x - min_x) / nx
    y_step = (max_y - min_y) / ny
    
    rects = []
    for i in range(nx):
        for j in range(ny):
            x1 = min_x + i * x_step
            y1 = min_y + j * y_step
            x2 = x1 + x_step
            y2 = y1 + y_step
            rects.append((x1, y1, x2, y2))
    return rects

def search_category_by_rect(rect, category_code="MT1", size=15):
    """하나의 rect에서 끝까지 페이지 반복"""
    rows = []
    min_x, min_y, max_x, max_y = rect
    rect_str = f"{min_x},{min_y},{max_x},{max_y}"
    
    page = 1
    last_meta = None
    
    while True:
        params = {
            "category_group_code": category_code,
            "rect": rect_str,
            "page": page,
            "size": size
        }
        
        res = requests.get(url, headers=headers, params=params, timeout=5)
        data = res.json()
        
        documents = data.get("documents", [])
        meta = data.get("meta", {})
        last_meta = meta
        
        rows.extend(documents)
        
        print(f"  rect={rect_str} | page={page} | total_count={meta.get('total_count')} | pageable_count={meta.get('pageable_count')} | is_end={meta.get('is_end')}")
        
        if meta.get("is_end", True):
            break
            
        page += 1
        time.sleep(0.2)
    
    return rows, last_meta

all_rows = []

for region_name, bbox in regions.items():
    print(f"\n===== {region_name} 수집 시작 =====")
    
    # 지역별 기본 분할 수
    # 서울/인천/큰 도시는 더 잘게
    if region_name == "서울":
        nx, ny = 5, 5
    elif region_name in ["인천", "성남", "고양", "용인", "남양주"]:
        nx, ny = 3, 3
    else:
        nx, ny = 2, 2
    
    rects = split_bbox(*bbox, nx=nx, ny=ny)
    
    for idx, rect in enumerate(rects, start=1):
        print(f"\n[{region_name}] 격자 {idx}/{len(rects)}")
        rows, meta = search_category_by_rect(rect, category_code="MT1", size=15)
        
        for doc in rows:
            doc["수집지역"] = region_name
            doc["격자번호"] = idx
            all_rows.append(doc)
        
        time.sleep(0.3)

df = pd.DataFrame(all_rows)

if not df.empty:
    # id 기준 중복 제거
    df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

print("\n최종 행 수:", len(df))
print(df[["place_name", "road_address_name", "x", "y", "수집지역"]].head())

# 저장
df.to_csv("kakao_mt1_split_result.csv", index=False, encoding="utf-8-sig")


===== 서울 수집 시작 =====

[서울] 격자 1/25
  rect=126.764,37.425,126.84779999999999,37.480199999999996 | page=1 | total_count=16 | pageable_count=16 | is_end=False
  rect=126.764,37.425,126.84779999999999,37.480199999999996 | page=2 | total_count=16 | pageable_count=16 | is_end=True

[서울] 격자 2/25
  rect=126.764,37.480199999999996,126.84779999999999,37.535399999999996 | page=1 | total_count=26 | pageable_count=26 | is_end=False
  rect=126.764,37.480199999999996,126.84779999999999,37.535399999999996 | page=2 | total_count=26 | pageable_count=26 | is_end=True

[서울] 격자 3/25
  rect=126.764,37.535399999999996,126.84779999999999,37.590599999999995 | page=1 | total_count=24 | pageable_count=24 | is_end=False
  rect=126.764,37.535399999999996,126.84779999999999,37.590599999999995 | page=2 | total_count=24 | pageable_count=24 | is_end=True

[서울] 격자 4/25
  rect=126.764,37.5906,126.84779999999999,37.6458 | page=1 | total_count=20 | pageable_count=20 | is_end=False
  rect=126.764,37.5906,126.8477999999999

In [35]:
df = df.rename(columns={'place_name':'지점','address_name' : '지번주소', 'road_address_name' : '도로명주소', 'x':'x좌표값', 'y':'y좌표값'})

In [36]:
df = df[['지점', '지번주소', '도로명주소', 'x좌표값', 'y좌표값']]

In [ ]:
# df.to_csv('data_cleaning/[생활]_대형마트모음.csv')

In [42]:
df['도로명주소'].str.contains('인천').sum()

np.int64(138)

In [46]:
df[['place_name','category_name']].sample(30)

,place_name,category_name
555,성남농협 하나로마트 로컬푸드직매장 대왕점,"가정,생활 > 슈퍼마켓 > 대형슈퍼 > 하나로마트"
395,롯데슈퍼프레시 잠실3동점,"가정,생활 > 슈퍼마켓 > 대형슈퍼 > 롯데슈퍼프레시"
432,홈플러스익스프레스 중곡점,"가정,생활 > 슈퍼마켓 > 대형슈퍼 > 홈플러스익스프레스"
953,롯데슈퍼프레시 남양주진접점,"가정,생활 > 슈퍼마켓 > 대형슈퍼 > 롯데슈퍼프레시"
250,롯데슈퍼프레시 고양삼송점,"가정,생활 > 슈퍼마켓 > 대형슈퍼 > 롯데슈퍼프레시"
979,다드림마트 오산점,"가정,생활 > 대형마트"
634,홈플러스익스프레스 망우점,"가정,생활 > 슈퍼마켓 > 대형슈퍼 > 홈플러스익스프레스"
355,GS더프레시 종로구기점,"가정,생활 > 슈퍼마켓 > 대형슈퍼 > GS더프레시"
865,보배로이 판교플래그십스토어,"가정,생활 > 대형마트"
401,GS더프레시 압구정점,"가정,생활 > 슈퍼마켓 > 대형슈퍼 > GS더프레시"
